# <font color="steelblue">21. Probabilidad condicionada e independencia</font>

*Unidad 4. Probabilidad* · Sesión teórica 21 de 30

**Adaptación para el Grado en Periodismo**: Fernando Borrás Rocher (f.borras@umh.es)

**Basado en SIMUSTAT**, de Fernando Borrás, Federico Botella, Inés Hernández, M.ª Asunción Martínez Mayoral, Josep Moltó y Javier Morales.
Departamento de Estadística, Matemáticas e Informática. Universidad Miguel Hernández de Elche.

**Asignatura**: Estadística (1200), 1.º curso, Grado en Periodismo.

**Licencia**: <a rel="license" href="http://creativecommons.org/licenses/by-sa/4.0/"><img alt="CC BY-SA 4.0" style="border-width:0" src="https://i.creativecommons.org/l/by-sa/4.0/88x31.png" /></a> Creative Commons Atribución-CompartirIgual 4.0.

> ⚠️ **Los datos de este curso son simulados.** Imitan la forma de fuentes reales (INE, CIS, audímetros, agencias de verificación), pero ninguna cifra es real. Úsalos para aprender; para publicar, ve siempre a la fuente oficial.

## <font color='steelblue'>Antes de empezar: configura el cuaderno</font>

Ejecuta **una vez** la celda de abajo (botón ▶). Instala la librería de las preguntas, descarga los datos (`verificaciones.csv`, `encuesta_medios.csv`) y las preguntas de autoevaluación. Si todo va bien verás **`Cuaderno configurado`**.

> 💡 Si Colab se desconecta, los ficheros se borran: vuelve a ejecutar esta celda.

In [ ]:
#@title <font color="steelblue" size="+1"> Ejecutar configuración { display-mode: "form" }
import os, sys, json, base64, subprocess, urllib.request

RAW = "https://raw.githubusercontent.com/fborrasumh/statperiodismo/main/"
DATOS = ["verificaciones.csv", "encuesta_medios.csv"]
PREGUNTAS = ["q_P21_1.json", "q_P21_2.json", "q_P21_3.json", "q_P21_4.json"]

try:
    from jupyterquiz import display_quiz
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "jupyterquiz==2.9.6.4"], check=True)
    from jupyterquiz import display_quiz

for carpeta, ficheros in (("datos/", DATOS), ("autoeval/", PREGUNTAS)):
    for f in ficheros:
        if not os.path.exists(f):
            urllib.request.urlretrieve(RAW + carpeta + f, f)

def pregunta(fichero):
    """Muestra una pregunta interactiva de autoevaluación."""
    display_quiz(json.loads(base64.b64decode(open(fichero, "rb").read()).decode("utf8")))

print("Cuaderno configurado")

## <font color='steelblue'>Introducción</font>

«El 70 % de los bulos se difunde por WhatsApp» y «el 70 % de lo que circula por WhatsApp es un bulo» son frases muy distintas, aunque usen el mismo número. La **probabilidad condicionada** formaliza la idea de «porcentaje sobre un grupo» que viste con las tablas de contingencia, y permite decidir si dos características son **independientes**.

**Qué necesitas saber antes**: los cuadernos 12 (tablas de contingencia) y 20.

## <font color='steelblue'>Qué vas a aprender</font>

Al terminar este cuaderno sabrás:

1. Calcular probabilidades condicionadas a partir de una tabla.
2. Distinguir $P(A \mid B)$ de $P(B \mid A)$ y reconocer la falacia del fiscal.
3. Usar la regla del producto.
4. Comprobar si dos sucesos son independientes.

## <font color='steelblue'>Contenidos</font>

1. Probabilidad condicionada
2. La falacia del fiscal
3. Regla del producto
4. Independencia

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (10, 4)
pd.set_option('display.max_columns', 20)

## <font color='steelblue'>1. Probabilidad condicionada</font>

In [ ]:
verif = pd.read_csv("verificaciones.csv")
verif["es_bulo"] = np.where(verif["veredicto"] == "Verdadero", "No", "Sí")   # falso o engañoso
tabla = pd.crosstab(verif["plataforma"], verif["es_bulo"], margins=True)
tabla

La **probabilidad de A condicionada a B** es la probabilidad de A **sabiendo que** ha ocurrido B:

$$P(A \mid B) = \frac{P(A \text{ y } B)}{P(B)}$$

En una tabla: los casos con A y B, divididos por los casos con B. Es exactamente un **porcentaje por fila** (o por columna).

In [ ]:
wa = verif[verif["plataforma"] == "WhatsApp"]
p_bulo_dado_wa = (wa["es_bulo"] == "Sí").mean()
print(f"P(bulo | WhatsApp) = {p_bulo_dado_wa:.3f}")

bulos = verif[verif["es_bulo"] == "Sí"]
p_wa_dado_bulo = (bulos["plataforma"] == "WhatsApp").mean()
print(f"P(WhatsApp | bulo) = {p_wa_dado_bulo:.3f}")

**Cómo leerlo**: de los contenidos de WhatsApp revisados, más de 9 de cada 10 eran bulos; pero de todos los bulos, solo un tercio venía de WhatsApp. Las dos cifras son ciertas y responden a preguntas distintas.

### <font color='steelblue'>Autoevaluación</font>

Ejecuta la celda y escoge la respuesta correcta (**solo hay una**):

In [ ]:
#@title <font color='steelblue'> Pregunta interactiva { display-mode: "form" }
pregunta("q_P21_1.json")

## <font color='steelblue'>2. La falacia del fiscal</font>

Confundir $P(A \mid B)$ con $P(B \mid A)$ se llama **falacia del fiscal**, porque aparece en juicios: «la probabilidad de que el ADN coincida por casualidad es de 1 entre 10.000» **no** significa que la probabilidad de que el acusado sea inocente sea de 1 entre 10.000.

Ejemplo numérico: en una ciudad de 500.000 habitantes, si la coincidencia casual es de 1/10.000, unas **50 personas** coincidirían por azar. Sin más pruebas, el acusado es solo uno de ellos.

In [ ]:
poblacion = 500_000
p_coincidencia_inocente = 1 / 10_000
coincidencias_casuales = poblacion * p_coincidencia_inocente
print("Personas que coincidirían por azar:", int(coincidencias_casuales))
print(f"P(culpable | coincide), si solo hay un culpable y nada más: {1 / (coincidencias_casuales + 1):.3f}")

### <font color='steelblue'>Autoevaluación</font>

Ejecuta la celda y escoge la respuesta correcta (**solo hay una**):

In [ ]:
#@title <font color='steelblue'> Pregunta interactiva { display-mode: "form" }
pregunta("q_P21_2.json")

## <font color='steelblue'>3. Regla del producto</font>

Despejando la fórmula de la condicionada se obtiene la **regla del producto**:

$$P(A \text{ y } B) = P(B) \cdot P(A \mid B)$$

Sirve para encadenar probabilidades: si el 22 % de las personas se informa por redes y, de ellas, el 65 % comparte noticias, el 0,22 × 0,65 ≈ 14 % de **toda** la población se informa por redes y comparte.

In [ ]:
enc = pd.read_csv("encuesta_medios.csv")
p_redes = (enc["medio_principal"] == "Redes sociales").mean()
p_comparte_dado_redes = (enc.loc[enc["medio_principal"] == "Redes sociales", "comparte_noticias"] == "Sí").mean()
print(f"P(redes) × P(comparte | redes) = {p_redes:.3f} × {p_comparte_dado_redes:.3f} = {p_redes * p_comparte_dado_redes:.3f}")
print("Comprobación directa:", round(((enc["medio_principal"] == "Redes sociales") & (enc["comparte_noticias"] == "Sí")).mean(), 3))

### <font color='steelblue'>Autoevaluación</font>

Ejecuta la celda y escribe el número y pulsa Enter:

In [ ]:
#@title <font color='steelblue'> Pregunta interactiva { display-mode: "form" }
pregunta("q_P21_3.json")

## <font color='steelblue'>4. Independencia</font>

Dos sucesos son **independientes** si saber que ha ocurrido uno no cambia la probabilidad del otro:

$$P(A \mid B) = P(A) \quad \Longleftrightarrow \quad P(A \text{ y } B) = P(A) \cdot P(B)$$

En una tabla: si las variables fueran independientes, los porcentajes por fila serían **iguales** al porcentaje del total.

In [ ]:
print("P(bulo) general:", round((verif["es_bulo"] == "Sí").mean(), 3))
(pd.crosstab(verif["plataforma"], verif["es_bulo"], normalize="index")["Sí"]).round(3)

In [ ]:
print("P(bulo) general:", round((verif["es_bulo"] == "Sí").mean(), 3))
(pd.crosstab(verif["tema"], verif["es_bulo"], normalize="index")["Sí"]).round(3)

**Cómo leerlo**: la probabilidad de bulo cambia bastante según la plataforma (no parecen independientes), pero apenas cambia según el tema (parecen independientes). En datos de muestras nunca saldrán exactamente iguales; en el cuaderno 30 verás cómo decidir si la diferencia es mayor de lo que produciría el azar.

### <font color='steelblue'>Autoevaluación</font>

Ejecuta la celda y escoge la respuesta correcta (**solo hay una**):

In [ ]:
#@title <font color='steelblue'> Pregunta interactiva { display-mode: "form" }
pregunta("q_P21_4.json")

## <font color='steelblue'>Reto aprendiz</font>

Resuelve estos ejercicios en la celda vacía de debajo de cada enunciado. Si te atascas, vuelve a la sección indicada. La solución está plegada: **inténtalo antes de mirarla**.

### **Ejercicio 1** *(sección 1)*

Calcula P(bulo | formato = Audio) y P(Audio | bulo).

<details><summary>Ver solución</summary>

```python
print((verif.loc[verif["formato"] == "Audio", "es_bulo"] == "Sí").mean())
print((verif.loc[verif["es_bulo"] == "Sí", "formato"] == "Audio").mean())
```
</details>

### **Ejercicio 2** *(sección 2)*

Redacta dos titulares, uno con cada probabilidad del ejercicio anterior, que no se puedan confundir.

<details><summary>Ver idea clave</summary>

Por ejemplo: «Más de 9 de cada 10 audios virales revisados eran falsos o engañosos» frente a «Solo 1 de cada 10 bulos verificados era un audio».
</details>

### **Ejercicio 3** *(sección 3)*

Con la encuesta, calcula por la regla del producto la probabilidad de que una persona tenga estudios universitarios y pague por noticias. Compruébalo directamente.

<details><summary>Ver solución</summary>

```python
pu = (enc["estudios"] == "Universitarios").mean()
pp_u = (enc.loc[enc["estudios"] == "Universitarios", "suscripcion_pago"] == "Sí").mean()
print(pu * pp_u, ((enc["estudios"] == "Universitarios") & (enc["suscripcion_pago"] == "Sí")).mean())
```
</details>

### **Ejercicio 4** *(sección 4)*

¿Parecen independientes la provincia y haber creído un bulo en la encuesta?

<details><summary>Ver solución</summary>

```python
print((enc["ha_creido_bulo"] == "Sí").mean())
pd.crosstab(enc["provincia"], enc["ha_creido_bulo"], normalize="index")
```
</details>

## <font color='steelblue'>Resumen: ideas clave</font>

- $P(A \mid B)$ es la probabilidad de A **dentro del grupo B**: un porcentaje por fila o por columna.
- $P(A \mid B)$ y $P(B \mid A)$ suelen ser muy distintas; confundirlas es la **falacia del fiscal**.
- **Regla del producto**: $P(A \text{ y } B) = P(B) \cdot P(A \mid B)$.
- A y B son **independientes** si $P(A \mid B) = P(A)$: los porcentajes por fila son iguales al total.

## <font color='steelblue'>Glosario</font>

| Término | Significado |
| :-- | :-- |
| **Probabilidad condicionada** | Probabilidad de un suceso sabiendo que ha ocurrido otro. |
| **Falacia del fiscal** | Confundir P(A | B) con P(B | A). |
| **Regla del producto** | P(A y B) = P(B) · P(A | B). |
| **Independencia** | Situación en la que saber un suceso no cambia la probabilidad del otro. |

## <font color='steelblue'>Errores frecuentes y cómo resolverlos</font>

| Síntoma | Causa probable | Solución |
| :-- | :-- | :-- |
| `NameError: name 'pd' is not defined` | No se ejecutó la celda de importación | Ejecuta las celdas desde el principio, en orden |
| `FileNotFoundError` al leer un fichero | No se ejecutó la configuración o Colab se reinició | Ejecuta la celda *Configura el cuaderno* |
| La pregunta de autoevaluación no aparece | Falta la configuración | Ejecuta la celda de configuración y luego la pregunta |

## <font color='steelblue'>Para seguir</font>

En el cuaderno 22 usarás el teorema de Bayes para entender por qué un detector de bulos «con un 95 % de acierto» puede equivocarse mucho.

**Siguiente cuaderno**: [22. Teorema de Bayes: ¿cuánto acierta un detector de bulos?](https://colab.research.google.com/github/fborrasumh/statperiodismo/blob/main/cuadernos/P22_teorema_de_bayes.ipynb)